# DDRI bike_change full161 기본피처 정제 Run-All 노트북

이 노트북은 새 실험 폴더에서 `161개` 정본 데이터 기반 `bike_change` 정제 데이터를 생성합니다.


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import time

import pandas as pd
from IPython.display import Markdown, display

root_candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
script_name = "02_ddri_bike_change_full161_base_dataset_builder.py"

script_path = None
for base in root_candidates:
    candidate = (base / "works" / "08_prediction_bike_change_full161_base" / script_name).resolve()
    if candidate.exists():
        script_path = candidate
        break
    candidate = (base / script_name).resolve()
    if candidate.exists():
        script_path = candidate
        break

if script_path is None:
    found = list(Path.cwd().rglob(script_name))
    if found:
        script_path = found[0].resolve()

if script_path is None:
    raise FileNotFoundError("실행 스크립트를 찾지 못했습니다.")

work_dir = script_path.parent
root_dir = work_dir.parents[1]
source_dir = root_dir / "3조 공유폴더" / "군집별 데이터_전체 스테이션" / "full_data"
print(f"script_path={script_path}")
print(f"source_dir={source_dir}")


In [ ]:
env = os.environ.copy()
env.setdefault("PYTHONUNBUFFERED", "1")
t0 = time.perf_counter()
process = subprocess.Popen(
    [sys.executable, "-u", str(script_path)],
    cwd=str(root_dir),
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in process.stdout:
    print(line, end="")
return_code = process.wait()
elapsed = time.perf_counter() - t0
print(f"총 실행시간: {elapsed:,.2f}초")
if return_code != 0:
    raise RuntimeError(f"정제 스크립트 실행 실패: return_code={return_code}")


In [ ]:
output_root = root_dir / "3조 공유폴더" / "군집별 데이터_전체 스테이션" / "bike_change_full161_base_outputs"
output_data_dir = output_root / "data"
output_report_dir = output_root / "reports"
train_out = output_data_dir / "ddri_prediction_bike_change_full161_base_train_2023_2024.csv"
test_out = output_data_dir / "ddri_prediction_bike_change_full161_base_test_2025.csv"
summary_out = output_data_dir / "ddri_prediction_bike_change_full161_base_feature_summary.csv"
meta_out = output_report_dir / "ddri_prediction_bike_change_full161_base_feature_meta.json"

train_df = pd.read_csv(train_out, nrows=5)
test_df = pd.read_csv(test_out, nrows=5)
summary_df = pd.read_csv(summary_out)
meta = json.loads(meta_out.read_text(encoding="utf-8"))

display(Markdown("### train 샘플"))
display(train_df)
display(Markdown("### test 샘플"))
display(test_df)
display(Markdown("### feature summary"))
display(summary_df)
display(Markdown(f"**train_rows_after_target**: `{meta['train_rows_after_target']}`  \n**test_rows_after_target**: `{meta['test_rows_after_target']}`  \n**seasonal_groups_from_train_2023**: `{meta['seasonal_groups_from_train_2023']}`  \n**output_root**: `{output_root}`"))
